# Bài 31 — chuỗi luyện Easy · LeetCode #160 — Intersection of Two Linked Lists

**Giáo trình Jupyter Notebook tiếng Việt cho người mới bắt đầu · Python 3**  
**Link đề:** https://leetcode.com/problems/intersection-of-two-linked-lists/ · **Mức độ:** Easy · **Chủ điểm:** Linked List · Two Pointers · tham chiếu đối tượng

**Kết quả bạn đạt được sau bài học**
1. Phát biểu được chính xác input, output, và những điều kiện đề bảo đảm; phân biệt dữ liệu thật với ví dụ mô phỏng.
2. Tự diễn giải công cụ Python liên quan bằng chương trình mini; không ghi nhớ cú pháp một cách máy móc.
3. Tự nghĩ ra một cách giải dễ hiểu, thấy rõ điểm hạn chế, rồi suy ra cách chính tốt hơn.
4. Lập bảng và chạy chương trình `trace_*` để thấy trạng thái thay đổi từng bước, kể cả nhánh đặc biệt.
5. Giải thích được tính đúng bằng **bất biến (invariant)**, tự suy luận Time/Space Complexity.
6. Viết lại `class Solution` để nộp LeetCode và tự giải 6 bài tập, có gợi ý ba mức và đáp án cuối.

> **Cách học:** Đọc một markdown → tự dự đoán kết quả code → chạy cell → so sánh → tự giải thích vì sao. Các cell TODO chỉ định nghĩa hàm, không được chạy tự động.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho hai đầu danh sách liên kết đơn `headA`, `headB`. Hãy trả về **chính đối tượng node** đầu tiên mà hai danh sách cùng trỏ tới; nếu hai danh sách hoàn toàn rời nhau, trả `None` (`null` trên LeetCode). Hai danh sách không chứa chu trình và **không được thay đổi cấu trúc** sau lời gọi.

Điểm dễ nhầm: `ListNode(val=8)` trong A và `ListNode(val=8)` trong B chưa chắc giao nhau. Chỉ giao nhau khi đó là **cùng một node trong bộ nhớ**, nghĩa là `a is b`, từ đó toàn bộ phần đuôi sau node ấy cũng là chung.

Ví dụ 1: A = `4 → 1 → (8 → 4 → 5)`; B = `5 → 6 → 1 → (8 → 4 → 5)`; ba nút trong ngoặc là **dùng chung đối tượng**. Kết quả trả về node có `val=8`. Ví dụ 2: hai danh sách có các giá trị trùng nhưng từng node khác nhau → `None`. Ràng buộc của đề được kiểm trên hình thức node; `intersectVal/skipA/skipB` thuộc custom judge, không phải tham số của hàm bạn viết.


## 2. Thiết lập môi trường để chạy notebook

Notebook dùng **Python 3** chuẩn, không cần cài thêm package. Hãy chạy **từ trên xuống dưới** để các tên hàm đã được định nghĩa trước khi gọi. Không copy `print(...)` vào LeetCode; các dòng in ở đây chỉ phục vụ việc học.

**Phân biệt ba lớp:** (1) demo nhỏ để hiểu công cụ, (2) hàm minh họa giải bài trong notebook, (3) `class Solution` đúng chữ ký để nộp trên trang đề. Với bài #160, ta sẽ học kỹ thuật **Linked List · Two Pointers · tham chiếu đối tượng** theo đúng thứ tự này.

Từng code cell sau đây cần cho ra output ngắn và có thể kiểm tra bằng mắt. Trong cell kiểm thử, `assert` không có output nghĩa là điều kiện đã đúng; nếu sai, Python báo `AssertionError`.


In [1]:
from __future__ import annotations
from typing import Optional
import random

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def __repr__(self):
        return f"Node({self.val})"

def make_list(values, tail=None):
    head = tail
    for x in reversed(values):
        head = ListNode(x, head)
    return head

def make_case(a_prefix, b_prefix, common_values):
    common = make_list(common_values)
    return make_list(a_prefix, common), make_list(b_prefix, common), common

print('Đã tạo ListNode, make_list, make_case.')


Đã tạo ListNode, make_list, make_case.


### 2.1. Node, `val`, `next` và cấu trúc móc nối

Một `ListNode` gồm giá trị `val` và một đường nối `next` tới node kế tiếp; `next=None` báo hết danh sách. Danh sách không phải `list` Python: ta không dùng `head[2]` hay gọi `len(head)`. Muốn đi từ node hiện tại sang node sau, viết `node = node.next`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [2]:
p = ListNode(4)
q = ListNode(1)
p.next = q
print(p.val, p.next.val, q.next)  # 4 1 None


4 1 None


### 2.2. Identity (`is`) khác so sánh giá trị

`x.val == y.val` chỉ hỏi hai số có giống nhau không. `x is y` hỏi **hai tên cùng tham chiếu tới một object** không. Bài #160 bắt buộc dùng identity. Nếu cùng `val=8` mà là hai node được tạo riêng, chúng không tạo giao điểm.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [3]:
a=ListNode(8); b=ListNode(8); alias=a
print('cùng số?', a.val==b.val)
print('cùng node?',a is b,'alias?',a is alias)


cùng số? True
cùng node? False alias? True


### 2.3. `None` và kiểm tra trước khi đọc thuộc tính

Một con trỏ có thể là `None`: nếu cố `None.next` sẽ bị `AttributeError`. Câu `p.next if p is not None else headB` nghĩa là khi còn node thì bước tiếp, khi đã hết danh sách thì chuyển sang đầu danh sách B. `is None` rõ ràng hơn suy đoán theo giá trị của `p.val`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [4]:
p=None
print('khi p=None:',p.next if p is not None else 'chuyển sang B')
p=ListNode(0)
print('nút có val=0 vẫn tồn tại:',p is not None)


khi p=None: chuyển sang B
nút có val=0 vẫn tồn tại: True


### 2.4. Vòng `while`, thao tác dừng và số bước

`while a is not b:` chỉ tiếp tục khi **chưa cùng node**. Nếu không giao nhau, cả hai cuối cùng cùng bằng `None`, vòng lặp kết thúc. Phải đặt điều kiện trước cập nhật để không đọc `.next` của `None`. Ta có thể đếm số vòng để theo dõi quá trình.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [5]:
x=make_list([2,3,4]); steps=0
while x is not None:
    print('bước',steps,'đang ở',x.val)
    x=x.next; steps+=1
print('đã đi qua',steps,'nút')


bước 0 đang ở 2
bước 1 đang ở 3
bước 2 đang ở 4
đã đi qua 3 nút


### 2.5. Tạo hai danh sách có đuôi CHUNG

Ta dựng `common` một lần, rồi nối cả hai nhánh riêng vào **cùng object** ấy. Nếu tạo hai danh sách bằng cách copy cùng giá trị, ta đã tạo hai đuôi khác nhau, không phải giao điểm. Hãy nhớ `make_case()` chỉ hỗ trợ demo, không phải code cần submit.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [6]:
ha,hb,common=make_case([4,1],[5,6,1],[8,4,5])
print('giao ở 8?',ha.next.next is common)
print('B tới 8?',hb.next.next.next is common)


giao ở 8? True
B tới 8? True


### 2.6. `set` lưu object và khái niệm baseline

Python cho phép lưu một `ListNode` mặc định vào `set` vì `ListNode` ở đây dùng cách so sánh theo identity (không ghi đè `__eq__`). Kiểm tra `node in seen` nhanh trung bình, nhưng set tốn bộ nhớ tỷ lệ với số node đã duyệt. Đây là cách dễ hiểu để có lời giải đúng ban đầu.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [7]:
ha,hb,common=make_case([1],[2],[9])
seen={ha,ha.next}
print('node chung trong set:',common in seen)
print('nút khác có cùng val?',ListNode(9) in seen)


node chung trong set: True
nút khác có cùng val? False


### 2.7. Hai con trỏ đi hai tuyến đường bằng nhau

Con trỏ `a` đi A→B; `b` đi B→A. Giả sử chiều dài riêng của A là x, riêng B là y, đuôi chung dài c. Mỗi con trỏ đi tổng x+y+c (trong trường hợp gặp tại đầu đuôi chung), nên độ lệch chiều dài ban đầu được khử; không phải tính độ dài để căn chỉnh.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [8]:
a_len=2+3; b_len=3+3
print('hai tuyến tổng =',a_len+b_len)
print('A+B và B+A:',a_len+b_len, b_len+a_len)


hai tuyến tổng = 11
A+B và B+A: 11 11


## 3. Nền tảng tư duy — từ ví dụ nhỏ đến bài toán

Bảy thí nghiệm ở trên không phải những mẹo rời rạc: chúng chính là các khối xây dựng cho bài #160. Hãy kiểm tra lần lượt: **dữ liệu đang nằm ở đâu, biến nào lưu thông tin đã biết, bước nào khiến ta tiến tới đáp án?**

Khi học thuật toán, đừng chỉ hỏi “code này chạy gì?”; hãy hỏi **tại sao không cần nhớ phần thông tin đã bỏ đi** và vì sao ta không bỏ sót lời giải. Phần sau sẽ chạy một số ví dụ rất nhỏ trước khi ghép thành hàm hoàn chỉnh.


In [9]:
def walk_values(head, limit=20):
    values=[]
    while head is not None and len(values)<limit:
        values.append(head.val); head=head.next
    return values

ha,hb,shared=make_case([4,1],[5,6,1],[8,4,5])
print('A:',walk_values(ha),'B:',walk_values(hb),'chung:',shared)


A: [4, 1, 8, 4, 5] B: [5, 6, 1, 8, 4, 5] chung: Node(8)


### Câu hỏi gợi mở trước khi tối ưu

- Nếu A và B đều có node giá trị 8 nhưng không dùng chung object, kết quả là gì?
- Nếu A dài 5 node, B dài 6 node, có thể so từng node cùng vị trí từ đầu không?
- Có cách nào làm cho hai con trỏ đi số bước bằng nhau mà không tạo `set` và không sửa liên kết?

Chưa cần thuộc tên kỹ thuật **Linked List · Two Pointers · tham chiếu đối tượng**. Trước tiên hãy tưởng tượng bạn chỉ có bút và giấy: bạn đọc input theo thứ tự nào, giữ lại những gì, điều gì buộc bạn dừng? Câu trả lời sẽ trở thành các biến và điều kiện `if`/`while`.


### Quy ước chạy cell và cách tự kiểm tra

- Đọc thứ tự từ trên xuống: các hàm helper trong phần này chỉ dùng để dựng dữ liệu hoặc hiển thị; đáp án submit sẽ không phụ thuộc chúng.
- `print` là quan sát, `return` là kết quả hàm. Dùng `assert actual == expected` để kiểm chứng, không thay thế một phép kiểm bằng việc nhìn thấy chữ PASS.
- Khi thấy nhiều thuật toán, chúng có **tên hàm khác nhau** (`baseline_*`, `solve_*`, `trace_*`) để tránh ghi đè logic. Không thay đổi input trừ khi đề cho phép.
- Đừng chạy hàm TODO trước khi tự hoàn thiện: hàm được để `pass`/`NotImplementedError` có chủ đích.

**Nguyên tắc quan trọng:** ví dụ và test là bằng chứng thử nghiệm; lập luận về bất biến mới giúp giải thích vì sao thuật toán đúng cho mọi input hợp lệ.


### 3.1. Theo dõi một node bằng identity

Ta cần phân biệt “đang ở giá trị 8” với “đang ở cùng địa chỉ node 8”. Một danh sách có thể chứa nhiều số trùng; chỉ `is` phát hiện đúng giao điểm. Vẽ A/B thành hai đường cùng đổ về một chiếc cầu chung giúp trực giác rõ hơn.

Chạy đoạn code dưới rồi thử tự diễn đạt **trạng thái trước – thao tác – trạng thái sau**. Nếu chỉ nhớ kết quả in ra mà không giải thích được sự chuyển đổi thì nên đọc lại ví dụ.


In [10]:
ha,hb,common=make_case([4,1],[5,6,1],[8,4,5])
a=ha.next.next; b=hb.next.next.next
print('A gặp:',a,'B gặp:',b,'same object?',a is b)
assert a is b is common


A gặp: Node(8) B gặp: Node(8) same object? True


### 3.2. Nhận ra tuyến A+B bằng các nhãn vị trí

Dùng các nhãn khác nhau cho phần riêng của hai danh sách và một nhãn chung cho đuôi. Sau khi đi hết A, ta chuyển sang đầu B; sau khi đi hết B, chuyển sang đầu A. Hai tuyến có đúng cùng số node; chỉ khác thứ tự xuất phát.

Đây là bước nối giữa ví dụ nhỏ và code thuật toán: những biến xuất hiện trong cell tiếp theo sẽ tái xuất hiện trong `solve_*` hoặc `trace_*` ở dưới.


In [11]:
A=['A0','A1','C0','C1']; B=['B0','B1','B2','C0','C1']
print('tuyến a:',A+B)
print('tuyến b:',B+A)
print('độ dài bằng nhau?',len(A+B)==len(B+A))


tuyến a: ['A0', 'A1', 'C0', 'C1', 'B0', 'B1', 'B2', 'C0', 'C1']
tuyến b: ['B0', 'B1', 'B2', 'C0', 'C1', 'A0', 'A1', 'C0', 'C1']
độ dài bằng nhau? True


## 4. Cách giải trực giác / Baseline

Cách dễ nghĩ: đi qua từng node A và lưu object vào một `set`. Sau đó đi qua B, node nào nằm trong set thì đó chính là node giao nhau đầu tiên trên B; vì sau điểm giao toàn bộ đuôi là chung nên điểm đầu tiên trên B cũng chính là điểm giao. Nếu hết B không thấy, trả `None`.

Tư duy này chuẩn xác, dễ kiểm chứng. Nhược điểm: có thể phải lưu đến m node khi A dài m, chưa đạt yêu cầu nâng cao `O(1)` bộ nhớ.

Ta cố ý viết phiên bản **dễ nghĩ nhất trước**, dù đôi khi nó dùng thêm bộ nhớ hoặc thao tác dư thừa. Lợi ích của baseline: nhìn được định nghĩa bài toán thành code, và có một “đáp án chuẩn” để đối chiếu khi tối ưu.


In [12]:
def baseline_160(headA, headB):
    seen=set()
    p=headA
    while p is not None:
        seen.add(p)
        p=p.next
    p=headB
    while p is not None:
        if p in seen:
            return p
        p=p.next
    return None


In [13]:
for ap,bp,common in [([4,1],[5,6,1],[8,4,5]),([1,2],[1,2],[]),([],[],[7])]:
    a,b,expected=make_case(ap,bp,common)
    actual=baseline_160(a,b)
    print(walk_values(a),walk_values(b),'->',actual,'đúng?',actual is expected)
    assert actual is expected


[4, 1, 8, 4, 5] [5, 6, 1, 8, 4, 5] -> Node(8) đúng? True
[1, 2] [1, 2] -> None đúng? True
[7] [7] -> Node(7) đúng? True


### 4.1. Đọc baseline và nhận ra chi phí

Gọi m, n là số node hai danh sách. Duyệt A tối đa m bước, B tối đa n bước nên **Time O(m+n)**. Set giữ tối đa m tham chiếu nên **Extra Space O(m)**. Mặc dù set tra cứu trung bình O(1), tổng thời gian vẫn tuyến tính; không nói set luôn O(1) trong trường hợp xấu nhất của bảng băm. Cách tốt hơn phải bỏ set.

Không nên khẳng định nhanh/chậm chỉ dựa trên việc chạy vài ví dụ. Hãy **đếm bước**, chỉ ra `n` đang đo cái gì và xét input xấu nhất. Sau này khi thử tối ưu, kết quả phải vẫn khớp baseline với cùng một input.


In [14]:
a,b,c=make_case([1,2,3],[4,5],[99,100])
seen=[];p=a
while p:
    seen.append(p);p=p.next
print('baseline phải nhớ',len(seen),'tham chiếu, kể cả node chỉ thuộc A')
print('đáp án chuẩn:',baseline_160(a,b))


baseline phải nhớ 5 tham chiếu, kể cả node chỉ thuộc A
đáp án chuẩn: Node(99)


### 4.2. Đường vòng / phản ví dụ giúp thấy lỗi tư duy

Sai lầm: đi đồng bộ từng nút theo chỉ số và kết luận không giao nếu khác nhau. Khi hai danh sách có phần riêng dài khác nhau, cặp con trỏ ở cùng vị trí không nhất thiết cùng đến giao điểm. Cũng sai nếu so `a.val == b.val`: node khác identity có thể chứa cùng giá trị.

Hãy tìm một đầu vào **cực nhỏ** làm giả thuyết sai; dạng test này hiệu quả hơn việc chỉ chạy ngẫu nhiên dữ liệu dài.


In [15]:
a,b,common=make_case([1],[2,3],[9])
print('A:',walk_values(a),'B:',walk_values(b))
print('bước đầu giá trị:',a.val,b.val,'giao thật:',common)
p=ListNode(9);q=ListNode(9)
print('trùng value không phải giao:',p.val==q.val,p is q)


A: [1, 9] B: [2, 3, 9]
bước đầu giá trị: 1 2 giao thật: Node(9)
trùng value không phải giao: True False


## 5. Vì sao cần bước cải tiến?

Baseline đã có Time O(m+n), nhưng lưu nhiều node. Chỉ cần loại bỏ chênh lệch chiều dài hai phần riêng, hai con trỏ có thể chạm nhau ngay tại giao điểm mà không ghi nhớ các node trước đó. Tính dài trước rồi căn chỉnh cũng O(m+n), O(1); nhưng cách **chuyển đầu** ngắn gọn hơn và không cần hai vòng đếm riêng.

Ta không muốn “tối ưu bằng phép màu”. Mục tiêu là **chứng minh có thể bỏ một phần công việc** mà vẫn giữ thông tin đủ để quyết định đúng. Sau khi tìm thấy đại lượng/cấu trúc bắt buộc phải duy trì, ta mới viết thuật toán chính.


### Bước đột phá — Mỗi con trỏ lần lượt đi qua **cả hai danh sách**

Đặt a ở headA, b ở headB. Sau mỗi vòng, nếu còn node thì tiến `.next`, còn `None` thì chuyển sang đầu danh sách kia. Tổng quãng đường A+B bằng B+A; nếu có đuôi chung, hai con trỏ gặp tại đầu đuôi; nếu không có, hai con trỏ đồng thời đến `None`. Không có thao tác thay đổi `.next` nào.

**Tự kiểm tra:** Nếu làm thêm đúng một bước nữa, cái gì chắc chắn vẫn đúng? Nếu thuật toán dừng ngay bây giờ, vì sao kết quả nó trả là hợp lệ? Hai câu hỏi này chính là hạt nhân của bất biến và điều kiện dừng.


## 6. Thuật toán chính — mổ xẻ từng bước rồi ghép lại

Chỉ cần hai biến `a`, `b`. **Điều kiện lặp `a is not b` là thiết yếu**: nó phân biệt identity, bao gồm cả trường hợp `a is b is None`. Hai phép gán trong vòng lặp cần cùng logic chuyển đầu, khác danh sách đích. Không dùng `a.val` hay lấy node theo vị trí trong một Python list.

Trước tiên xem một phép biến đổi ngắn trong cell nhỏ, sau đó đọc hàm `solve_*` không có `print`. **Không nhảy thẳng đến class Solution**: phần luyện tay và trace phía dưới sẽ làm rõ từng điều kiện.


### 6.1. Đọc từng biến, không chỉ đọc tên hàm

- `a = headA`, `b = headB`: xuất phát hai đầu.
- `while a is not b`: nếu trùng node (hoặc cùng None) dừng ngay.
- `a = a.next if a is not None else headB`: a đi A rồi B.
- `b = b.next if b is not None else headA`: b đi B rồi A.
- `return a`: trả **object giao** hoặc None.

Số bước tối đa tuyến tính theo tổng độ dài danh sách; không có `set` và không sửa dữ liệu.

Đặc biệt để ý **thứ tự cập nhật**: đổi thứ tự hai dòng tưởng nhỏ có thể thay đổi ý nghĩa của bất biến. Nếu nhầm, hãy viết trạng thái trước và sau bằng một ví dụ ba phần tử / ba bước.


In [16]:
a,b,common=make_case([4],[5,6],[9])
print('A head:',a,'B head:',b)
print('a.next:',a.next,'b.next:',b.next)
print('sau khi đến cuối A, a chuyển sang:',b)


A head: Node(4) B head: Node(5)
a.next: Node(9) b.next: Node(6)
sau khi đến cuối A, a chuyển sang: Node(5)


In [17]:
def solve_160(headA: Optional[ListNode], headB: Optional[ListNode]) -> Optional[ListNode]:
    a, b = headA, headB
    while a is not b:
        a = a.next if a is not None else headB
        b = b.next if b is not None else headA
    return a


### 6.2. Tại sao các nhánh `if`/`while` được viết đúng thứ tự?

Nếu A có 5 node, B có 6 node, `a` ban đầu có thể “đi sớm” hơn `b` khi tới đuôi chung. Khi `a` chuyển sang B và `b` chuyển sang A, sự chênh lệch chiều dài được bù chính xác. Hai phép gán không ảnh hưởng nhau vì chúng cập nhật hai biến riêng. Nếu cả hai là None, while không chạy và trả None.

Trong một vòng lặp hãy đọc bằng câu tiếng Việt: “**Trước bước này ta đã biết ...; ta kiểm tra ...; nếu ... thì ...; sau bước này ...**”. Có thể thêm `print` để kiểm tra suy luận, nhưng hàm nộp bài phải trả **đúng kiểu** kết quả.


In [18]:
for ap,bp,cv in [([4,1],[5,6,1],[8,4,5]),([1],[2,3],[9]),([1],[2],[]),([],[],[])]:
    a,b,c=make_case(ap,bp,cv)
    result=solve_160(a,b)
    assert result is c
    print('A=',walk_values(a),'B=',walk_values(b),'->',result)


A= [4, 1, 8, 4, 5] B= [5, 6, 1, 8, 4, 5] -> Node(8)
A= [1, 9] B= [2, 3, 9] -> Node(9)
A= [1] B= [2] -> None
A= [] B= [] -> None


### 6.3. Kiểm tra mấu chốt bằng vài bước tay

Ở từng vòng, `a` và `b` đều đã tiến cùng số *bước* trên lộ trình A+B và B+A (có thể tại thời điểm đó hai lộ trình đang nằm ở các danh sách khác nhau). Sau khi hai con trỏ đã “trả” phần chênh riêng, chúng ở cùng khoảng cách còn lại đến đuôi chung. Không node nào bị thay đổi.

Một bất biến đúng phải có ba điều: **đúng lúc bắt đầu; vẫn đúng sau một lần cập nhật; khi dừng thì suy ra đáp án**. Dù chưa biết từ “invariant”, bạn có thể phát biểu tự nhiên như câu trên.


In [19]:
a,b,common=make_case([1],[2,3],[9,10]); p,q=a,b
for step in range(7):
    print('step',step,'a=',p,'b=',q,'same?',p is q)
    if p is q: break
    p=p.next if p is not None else b
    q=q.next if q is not None else a


step 0 a= Node(1) b= Node(2) same? False
step 1 a= Node(9) b= Node(3) same? False
step 2 a= Node(10) b= Node(9) same? False
step 3 a= None b= Node(10) same? False
step 4 a= Node(2) b= None same? False
step 5 a= Node(3) b= Node(1) same? False
step 6 a= Node(9) b= Node(9) same? True


## 7. TRACE từng bước — nhìn thấy trạng thái biến

Ta sẽ dùng chương trình `trace_*` khác với lời giải chính: `solve_*` phục vụ submit và test; `trace_*` chỉ phục vụ việc **quan sát**. Trace phải cho thấy dữ liệu đầu vào, biến *trước khi cập nhật*, điều kiện/nhánh đang xét, biến *sau khi cập nhật*, và điều kiện dừng.

Ví dụ A: `4 → 1 → 8 → 4 → 5`, B: `5 → 6 → 1 → 8 → 4 → 5`, trong đó cùng dùng node `8` trở về sau. Ở trace ta in `A0, B0, C0,...` qua `id(node)` để tránh nhầm những node có `.val` trùng nhau.


In [20]:
def trace_160(headA, headB, limit=30):
    a,b=headA,headB
    for step in range(limit):
        av='None' if a is None else f'{a.val}@{id(a)%10000}'
        bv='None' if b is None else f'{b.val}@{id(b)%10000}'
        print(f'{step:2d} | a={av:12s} | b={bv:12s} | gặp nhau={a is b}')
        if a is b:
            print('DỪNG, trả:',a)
            return a
        a=a.next if a is not None else headB
        b=b.next if b is not None else headA
    raise RuntimeError('Vượt limit trace (kiểm tra input có cycle?)')

a,b,c=make_case([4,1],[5,6,1],[8,4,5]); assert trace_160(a,b) is c


 0 | a=4@9552       | b=5@9840       | gặp nhau=False
 1 | a=1@8976       | b=6@9744       | gặp nhau=False
 2 | a=8@9264       | b=1@9648       | gặp nhau=False
 3 | a=4@7248       | b=8@9264       | gặp nhau=False
 4 | a=5@8400       | b=4@7248       | gặp nhau=False
 5 | a=None         | b=5@8400       | gặp nhau=False
 6 | a=5@9840       | b=None         | gặp nhau=False
 7 | a=6@9744       | b=4@9552       | gặp nhau=False
 8 | a=1@9648       | b=1@8976       | gặp nhau=False
 9 | a=8@9264       | b=8@9264       | gặp nhau=True
DỪNG, trả: Node(8)


### 7.1. Trace dạng bảng — diễn giải cột và thứ tự cập nhật

Đọc bảng ở cell trên: cột `step` đếm số lần đã cập nhật; hai cột `a` và `b` là node **trước** lần cập nhật mới; `gặp nhau` kiểm tra identity. Nhãn `@...` chỉ giúp thấy hai node khác nhau dù cùng `.val`. Điểm dừng đúng là cùng node chung, không phải cùng số.

Khi đọc bảng, đừng coi một hàng là một đáp án; một hàng biểu diễn **một phép chuyển trạng thái**. Hãy tự che cột cuối, dự đoán hành động, rồi đối chiếu với print phía dưới.


In [21]:
a,b,c=make_case([1],[2,3],[9])
print('TRƯỜNG HỢP CHÊNH ĐỘ DÀI:')
assert trace_160(a,b) is c


TRƯỜNG HỢP CHÊNH ĐỘ DÀI:
 0 | a=1@7152       | b=2@128        | gặp nhau=False
 1 | a=9@8592       | b=3@32         | gặp nhau=False
 2 | a=None         | b=9@8592       | gặp nhau=False
 3 | a=2@128        | b=None         | gặp nhau=False
 4 | a=3@32         | b=1@7152       | gặp nhau=False
 5 | a=9@8592       | b=9@8592       | gặp nhau=True
DỪNG, trả: Node(9)


### 7.2. Một nhánh đặc biệt mà cách học thuộc code dễ bỏ lỡ

Không giao nhau nhưng có giá trị trùng. Sau khi lần lượt đi A+B và B+A, cả hai đến None. Chúng **không bao giờ** gặp nhau ở node 9 được tạo riêng. Test này phân biệt code dùng `is` với code dùng `.val` rất hiệu quả.

Câu hỏi sau khi chạy: **Nếu bỏ điều kiện xử lý nhánh này, input nhỏ nhất nào trả sai?**


In [22]:
a=make_list([9,8,7]); b=make_list([1,9,8,7])
assert trace_160(a,b) is None
print('Cùng các số 9,8,7 nhưng không chia sẻ node.')


 0 | a=9@9840       | b=1@9552       | gặp nhau=False
 1 | a=8@8400       | b=9@9648       | gặp nhau=False
 2 | a=7@7248       | b=8@9744       | gặp nhau=False
 3 | a=None         | b=7@7152       | gặp nhau=False
 4 | a=1@9552       | b=None         | gặp nhau=False
 5 | a=9@9648       | b=9@9840       | gặp nhau=False
 6 | a=8@9744       | b=8@8400       | gặp nhau=False
 7 | a=7@7152       | b=7@7248       | gặp nhau=False
 8 | a=None         | b=None         | gặp nhau=True
DỪNG, trả: None
Cùng các số 9,8,7 nhưng không chia sẻ node.


### 7.3. Trường hợp biên — không chỉ là dữ liệu “đẹp”

Các trường hợp khác thường: `headA=headB=None`; cả hai trỏ cùng đầu danh sách; một danh sách rỗng. Theo đề, danh sách không chứa cycle nên chỉ cần xét đường đi kết thúc tại None, không dùng Floyd cycle detection ở bài #141.

Trường hợp biên không phải ngoại lệ vô nghĩa: nó bộc lộ một giả định ẩn trong điều kiện lặp hoặc cách tính chỉ số. Chỉ thêm xử lý khi ràng buộc thực sự cho phép trường hợp đó.


In [23]:
print('hai danh sách rỗng:',solve_160(None,None))
a=make_list([1,2]);print('hai head cùng object:',solve_160(a,a))
print('một rỗng:',solve_160(a,None))
assert solve_160(a,a) is a and solve_160(a,None) is None


hai danh sách rỗng: None
hai head cùng object: Node(1)
một rỗng: None


### 7.4. Đối chiếu baseline và tối ưu trên một bộ input

Ta có hai thuật toán được suy ra độc lập: `baseline_160` lưu set node trong A; `solve_160` cân bằng quãng đường. Với một điểm giao tạo bằng `make_case`, kiểm tra **chính object** `actual is expected`, không dùng `==` trên value.

Nếu hai lời giải cho kết quả khác nhau, chưa kết luận ngay “baseline sai”; hãy in trạng thái tại input phản ví dụ, đối chiếu lại định nghĩa, rồi sửa mệnh đề bất biến.


In [24]:
for ap,bp,common_vals in [([],[],[]),([],[],[1]),([1,2],[3],[4]),([7],[7],[])]:
    a,b,c=make_case(ap,bp,common_vals)
    assert baseline_160(a,b) is solve_160(a,b) is c
    print('PASS',ap,bp,common_vals)


PASS [] [] []
PASS [] [] [1]
PASS [1, 2] [3] [4]
PASS [7] [7] []


## 8. Vì sao thuật toán đúng? — chứng minh ngắn nhưng đủ ý

**Bất biến:** Hai con trỏ đã đi cùng số bước trong hai chuỗi bước A+B và B+A. Cả hai lần lượt đi đủ các đoạn riêng có tổng độ dài `x+y`, do đó khi chúng có thể gặp trong đuôi chung, khoảng lệch ban đầu đã bị triệt tiêu. **Không thể gặp sai** ở node riêng nếu dùng `is`, vì các node riêng của hai list là object khác. **Có giao:** từ sau khi bù chênh, hai con trỏ tiến cùng tốc độ và gặp ở node chung đầu tiên. **Không giao:** mỗi con trỏ duyệt hết A+B hoặc B+A rồi cùng `None`. Vòng lặp dừng hữu hạn, trả node đúng hoặc None. Không liên kết nào bị sửa.

**Phân biệt:** `assert` minh họa “đã chạy đúng các ca này”, không chứng minh hết vô hạn input. Chứng minh dựa vào sự thật được giữ qua mọi lần lặp và điều kiện dừng; đó là kỹ năng sẽ dùng lại ở nhiều bài LeetCode khác.


## 9. Code nộp LeetCode (Python 3)

Đây là phiên bản **không in debug**, trả về đúng dữ liệu cần thiết, giữ đúng tên phương thức `getIntersectionNode` và tên tham số của đề. Khi submit, copy **chỉ** `class Solution` (và import nếu có dùng); với bài dùng `ListNode`, LeetCode đã cung cấp định nghĩa kiểu node.

Sau cell này, notebook sẽ gọi chính `Solution().getIntersectionNode(...)` bằng `assert` để kiểm tra cách đóng gói, tránh trường hợp hàm minh họa đúng nhưng chữ ký nộp sai.


In [25]:
class Solution:
    def getIntersectionNode(self, headA: Optional[ListNode], headB: Optional[ListNode]) -> Optional[ListNode]:
        a, b = headA, headB
        while a is not b:
            a = a.next if a is not None else headB
            b = b.next if b is not None else headA
        return a


## 10. Kiểm thử: mẫu đề, biên, phản ví dụ, kiểm thử đối chiếu

Ba nhóm bắt buộc: giao ở đuôi sâu, giao ngay head (hai head cùng object), và không giao dù giá trị node trùng. LeetCode yêu cầu **node**, không phải `.val` hay vị trí. Test identity vì vậy là điều kiện bắt buộc.

Với một ca test, viết rõ **input → expected → actual**. Ưu tiên các ca làm lộ bug thay vì lặp lại hai ví dụ thông thường. Test với input gốc theo đúng kiểu LeetCode, không chỉ kiểm tra các biến đã được biến đổi trong demo.


In [26]:
a,b,c=make_case([4,1],[5,6,1],[8,4,5]); assert Solution().getIntersectionNode(a,b) is c
a,b,c=make_case([1,9],[3],[2,4]); assert Solution().getIntersectionNode(a,b) is c
a,b,c=make_case([2,6,4],[1,5],[]); assert Solution().getIntersectionNode(a,b) is None
for a,b,c in [(None,None,None), (make_list([1]),None,None)]:
    assert Solution().getIntersectionNode(a,b) is c
shared=make_list([8,9]);assert Solution().getIntersectionNode(shared,shared) is shared
print('5 nhóm kiểm thử LeetCode: PASS')


5 nhóm kiểm thử LeetCode: PASS


### 10.1. Test hệ thống trên một miền nhỏ

Ta duyệt mọi độ dài phần riêng 0..4 của hai list, độ dài đuôi chung 0..3. Giá trị chọn sao cho có nhiều node chứa số giống nhau, kiểm tra identity chứ không phải value. Như vậy 5×5×4 = 100 trường hợp có cấu trúc khác nhau.

Với các input nhỏ, có thể duyệt toàn bộ cấu hình thỏa ràng buộc để tìm phản ví dụ. Nếu không thể duyệt tất cả do số lượng quá lớn, ta kiểm tra toàn bộ một miền con và nêu rõ phạm vi đó, **không gọi nó là chứng minh tổng quát**.


In [27]:
checked=0
for la in range(5):
    for lb in range(5):
        for lc in range(4):
            a,b,c=make_case([1]*la,[1]*lb,[1]*lc)
            assert solve_160(a,b) is baseline_160(a,b) is c
            assert Solution().getIntersectionNode(a,b) is c
            checked+=1
print('Kiểm tra vét cạn cấu trúc nhỏ:',checked,'PASS')


Kiểm tra vét cạn cấu trúc nhỏ: 100 PASS


### 10.2. Kiểm thử ngẫu nhiên **có seed** và so sánh với oracle

Tạo ngẫu nhiên chiều dài các đoạn riêng và đoạn chung, so lời giải tối ưu với oracle dùng set. Cần seed cố định vì cấu trúc tham chiếu mới được tạo mỗi lần; so sánh object `is` ổn định trong từng lần chạy.

`random.Random(seed)` khiến bộ test có thể chạy lại giống hệt. **Oracle** là một cách kiểm tra độc lập, thường chậm hơn nhưng dễ kiểm chứng. Không dùng hai code giống hệt nhau để đối chiếu — nếu cùng mắc lỗi, test sẽ cho cảm giác an toàn giả.


In [28]:
rng=random.Random(160); checked=0
for _ in range(400):
    la,lb,lc=(rng.randrange(30) for _ in range(3))
    a,b,c=make_case([rng.randrange(3) for _ in range(la)],
                    [rng.randrange(3) for _ in range(lb)],
                    [rng.randrange(3) for _ in range(lc)])
    assert solve_160(a,b) is baseline_160(a,b) is c
    checked+=1
print('Random oracle:',checked,'PASS')


Random oracle: 400 PASS


### 10.3. Tính chất khác của đáp án và dữ liệu đầu vào

Bài #160 cấm sửa cấu trúc: ghi lại `(id(node), id(node.next))` trước và sau. Thêm: nếu đổi `headA` ↔ `headB`, node giao đầu tiên vẫn như cũ; node riêng hai bên có thể cùng giá trị nhưng không được nhận nhầm.

Không chỉ hỏi “trả đúng số chưa?”; đôi khi đề còn yêu cầu **không sửa cấu trúc** hoặc có ràng buộc độ dài/bit. Kiểm thử thuộc tính giúp phát hiện lỗi chỉ xuất hiện ở nhóm input đặc biệt.


In [29]:
def link_snapshot_160(head):
    result=[]
    while head is not None:
        result.append((id(head),id(head.next)))
        head=head.next
    return result

a,b,c=make_case([5,6],[7],[8,9])
before=(link_snapshot_160(a),link_snapshot_160(b))
assert solve_160(a,b) is solve_160(b,a) is c
assert (link_snapshot_160(a),link_snapshot_160(b))==before
print('Không đổi next / hoán đổi A-B: PASS')


Không đổi next / hoán đổi A-B: PASS


## 11. Phân tích Time / Space Complexity

Đặt **m** là số node list A, **n** là số node list B. **Baseline:** Time O(m+n), Extra Space O(m) vì lưu node của A trong set. **Cách hai con trỏ:** mỗi con trỏ đi nhiều nhất O(m+n) bước, do vậy **Time O(m+n)**; chỉ tạo hai biến tham chiếu nên **Extra Space O(1)**. Không có cấu trúc output mới: trả một reference có sẵn hoặc None. Đừng nhầm “tổng hai lần đi” với O(mn): ta cộng độ dài, không lồng hai vòng.

Để tránh nói Big-O máy móc: chỉ rõ biến kích thước input, số vòng chạy, số bộ nhớ phụ thực sự cấp phát. Nếu đầu ra có độ dài bắt buộc, phân biệt **bộ nhớ phụ** với **bộ nhớ dành cho kết quả**; nếu 32 bit là hằng số, đừng vô tình coi đó là biến tăng vô hạn.


## 12. Sai lầm người mới hay mắc — code sai, vì sao sai, cách sửa

Ba lỗi thường thấy: so số thay vì node, quên chuyển đầu khi gặp None (bỏ mất cách cân bằng độ dài), hoặc thay đổi `.next` để nối hai danh sách thành một list. Bài này không được sửa cấu trúc input, kể cả sửa rồi “cố gắng sửa lại”.

Không lấy việc có kết quả đúng trên ví dụ đầu làm lý do bỏ qua các lỗi sau. Với từng lỗi, chỉ ra một input phản chứng; chạy đoạn minh họa riêng, sau đó đối chiếu với `solve_*` đã được test.


In [30]:
u=ListNode(8);v=ListNode(8)
print('Sai nếu dùng val:',u.val==v.val,'; đúng phải xem:',u is v)
assert (u.val==v.val) and (u is not v)


Sai nếu dùng val: True ; đúng phải xem: False


### 12.1. Lỗi thứ nhất: nhầm `node.val == other.val` với cùng node

Hai list có thể chứa cùng số nhưng không giao nhau. Cách sửa không phải thay `==` bằng `!=`; phải dùng `is` trực tiếp trên node, để so địa chỉ đối tượng được chia sẻ.

**Cách sửa:** quay lại định nghĩa và điều kiện dừng, không chỉ thêm một `if` để vá riêng input mẫu.


In [31]:
def bad_by_value_160(a,b):
    while a is not None and b is not None:
        if a.val==b.val:return a  # sai: a,b là object khác
        a=a.next;b=b.next
    return None
x=make_list([8]);y=make_list([8])
assert bad_by_value_160(x,y) is x and solve_160(x,y) is None
print('Phản ví dụ: cùng số 8 nhưng không có giao điểm')


Phản ví dụ: cùng số 8 nhưng không có giao điểm


### 12.2. Lỗi thứ hai: quên chuyển đầu khi con trỏ hết danh sách

Đọc song song từ đầu A và B không giải quyết việc hai list có độ dài khác nhau. Khi một con trỏ đến None, thay vì dừng lập tức, hãy chuyển nó qua đầu list còn lại; cả hai phải được cập nhật nhất quán.

**Cách sửa:** lập bảng một hoặc hai bước đầu để biết một phép tính dùng trạng thái *trước* hay *sau* cập nhật. Khi đã thấy lỗi, ghi lại một regression test để bug không quay lại.


In [32]:
def bad_parallel_160(a,b):
    while a and b:
        if a is b:return a
        a=a.next;b=b.next
    return None
x,y,c=make_case([1],[2,3],[9])
print('Duyệt song song sai:',bad_parallel_160(x,y),'| đúng:',solve_160(x,y))
assert bad_parallel_160(x,y) is None and solve_160(x,y) is c


Duyệt song song sai: None | đúng: Node(9)


## 13. Nhận diện pattern và tự suy ra kỹ thuật ở bài khác

Khi đề cho **hai chuỗi node nối đuôi**, cần tìm object chung, không cho sửa cấu trúc và đòi O(1) nhớ, hãy nghĩ đến **two pointers switching heads**. Một kiểu khác: đếm độ dài hai list, đưa con trỏ ở list dài đi trước độ lệch, rồi đồng bộ; có cùng O(m+n) / O(1). Không áp dụng trực tiếp nếu dữ liệu có chu trình, vì phép “đến None” không còn bảo đảm.

Một kỹ thuật không gắn với duy nhất bài #160: điều cần ghi nhớ là **dấu hiệu trong đề**, **trạng thái tối thiểu cần giữ**, **quy tắc cập nhật**, **khi nào dừng**, và **mức độ phức tạp có thể đạt được**. Thử tự nói thành 4–5 câu thay vì ghi nhớ code.


## 14. Biến thể — thay đổi yêu cầu để kiểm tra hiểu thật

Hai biến thể có ích: (1) trả vị trí giao đầu tiên tính từ mỗi head; (2) tìm số node của đuôi chung. Chúng đều dùng lại object giao đã xác định nhưng bổ sung cách đi/cách đếm; bài gốc vẫn chỉ yêu cầu trả node.

Nếu hiểu thuật toán, bạn có thể thay đổi mục tiêu mà vẫn suy ra cách làm. Phần này chỉ giới thiệu biến thể **có ý nghĩa và nằm ngoài yêu cầu gốc**; code submit phía trên vẫn bám đúng đề LeetCode.


In [33]:
a,b,c=make_case([1,2],[3],[8,9])
node=solve_160(a,b)
print('Node giao:',node,'giá trị:',node.val if node else None)


Node giao: Node(8) giá trị: 8


### 14.1. Biến thể thứ nhất: trả hai chỉ số vị trí của node giao trong A và B

Sau khi đã lấy node giao với O(1) extra, đi từ mỗi head đến node ấy để đếm index 0-based. Nếu không giao, trả `(-1,-1)` theo quy ước biến thể. Không so `.val` vì có thể có giá trị trùng trước giao.

Hãy xác định đầu ra mới có cần nhớ thêm trạng thái không. Nếu có, giải thích vì sao; nếu không, chỉ ra biến đang đủ dùng.


In [34]:
def intersection_positions_160(a,b):
    meet=solve_160(a,b)
    if meet is None:return (-1,-1)
    i=0;j=0;p=a;q=b
    while p is not meet:p=p.next;i+=1
    while q is not meet:q=q.next;j+=1
    return (i,j)
a,b,c=make_case([4,1],[5,6,1],[8,4])
assert intersection_positions_160(a,b)==(2,3)
print('Vị trí node giao:',intersection_positions_160(a,b))


Vị trí node giao: (2, 3)


### 14.2. Biến thể thứ hai: đếm độ dài phần đuôi dùng chung

Khi đã biết điểm giao `meet`, việc đi từ `meet` đến None không phụ thuộc đoạn riêng của A/B. Độ dài đuôi chung bằng số node còn lại. Nếu `meet=None`, độ dài đuôi chung là 0; không cần cấp thêm bộ nhớ.

Khi thay đổi ràng buộc đầu vào, các chứng minh cho bài gốc có thể **không còn đúng**; cần sửa chứng minh trước khi sửa code.


In [35]:
def common_tail_length_160(a,b):
    p=solve_160(a,b);ans=0
    while p is not None:
        ans+=1;p=p.next
    return ans
a,b,c=make_case([4,1],[5,6],[8,4,5])
assert common_tail_length_160(a,b)==3
print('Độ dài đuôi chung:',common_tail_length_160(a,b))


Độ dài đuôi chung: 3


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

**Bài 1 — dự đoán output:** Nếu `x=ListNode(7)`, `y=ListNode(7)`, `z=x`, hãy dự đoán `x is y`, `x is z`, `x.val == y.val`. **Đừng chạy ngay:** ghi kết quả dự đoán, rồi đối chiếu với code nền tảng.

**Bài 2 — tính tay / trace:** A = A0→C0→C1; B = B0→B1→C0→C1. Vẽ đường A+B và B+A; xác định node gặp đầu. Ghi ít nhất ba cột *trạng thái trước / thao tác / trạng thái sau*.

**Bài 3 — TODO viết lời giải chính:** hoàn thiện hàm dưới đây mà **không nhìn phần đáp án**.

**Bài 4 — biến thể:** thay đổi kiểu output hoặc điều kiện nhưng dùng lại mấu chốt thuật toán.

**Bài 5 — tự tạo kiểm thử:** nghĩ ít nhất 5 ca phân biệt được code đúng/sai, có ca biên và ca phản chứng.

**Bài 6 — mini challenge:** tự tổng quát hóa kỹ thuật, thử phân tích độ phức tạp và đối chiếu với baseline.

> Các hàm `exercise_*` bên dưới cố ý chưa có lời giải. Chúng **không được gọi tự động** khi chạy notebook. Đáp án chỉ xuất hiện ở mục 17.


### Bài 3 — TODO tự viết lời giải #160

Viết `exercise_intersection_160(headA,headB)` theo cách chuyển đầu. Ràng buộc: chỉ được dùng hai biến con trỏ, không tạo set, không sửa `.next`, trả node gốc hoặc None.

Gợi ý tối thiểu: trước khi viết, hãy ghi bất biến bằng một câu tiếng Việt; hàm bắt buộc trả đúng loại dữ liệu yêu cầu. Sau khi tự viết, bỏ `pass` và test thủ công với ví dụ và ca biên ở mục 10.


In [36]:
def exercise_intersection_160(headA, headB):
    # TODO 1: tạo hai con trỏ tại hai head
    # TODO 2: trong khi chưa cùng node, mỗi con trỏ đi tiếp hoặc chuyển đầu
    # TODO 3: trả object giao hoặc None
    pass


### Bài 4 — TODO biến thể

Viết hàm `exercise_first_value_160` trả giá trị `val` tại giao điểm; nếu không giao trả `None`. Đây là **biến thể**, không phải output của đề gốc. Giá trị có thể bằng 0, nên không dùng điều kiện `if result.val`.

Bạn được thay kiểu đầu ra nhưng không được thay định nghĩa bài gốc. Để hàm có chữ ký riêng và **không gọi tự động**; tạo dữ liệu thử sau khi đã có code.


In [37]:
def exercise_first_value_160(headA,headB):
    # TODO: gọi hoặc viết lại hai con trỏ; nếu node tồn tại, trả .val
    pass


### Bài 5 — TODO tự tạo test có chủ đích

Tạo ít nhất năm case: chung một head, chung đuôi dài 1, chỉ A rỗng, không giao nhưng giá trị trùng, hai prefix độ dài lệch nhau; dùng `is` kiểm tra node.

Thử đặt câu hỏi: một chương trình sai phổ biến có thể vẫn vượt qua ca nào? Cần thêm ca nào để khiến nó lộ sai? Viết kiểm thử `assert` trong hàm độc lập, sau đó chỉ gọi khi đã hoàn thiện hàm TODO 3, 4.


In [38]:
def exercise_tests_160():
    # TODO: tạo case bằng make_case rồi viết ít nhất 5 assert
    # Chỉ chạy sau khi hoàn thiện TODO 3/4.
    pass


### Bài 6 — mini challenge chuyển giao kỹ thuật

Không dùng switch-head; hãy tự viết cách **tính hai chiều dài**, đẩy con trỏ ở list dài đi trước hiệu số, rồi đi đồng tốc đến khi gặp. Đạt O(m+n) time và O(1) space, chứng minh vì sao không bỏ sót node giao.

Đây là bài nâng cao: bạn không cần giải nhanh. Hãy trình bày ý tưởng bằng tay, viết code sạch, tự mô phỏng input nhỏ rồi mới so sánh với đáp án tham khảo cuối notebook.


In [39]:
def exercise_align_length_160(headA,headB):
    # TODO: đếm độ dài hai list; căn chỉnh; so bằng `is`
    pass


## 16. Gợi ý ba mức — chỉ mở sau khi tự thử

**Bài 1:** Mức 1: hai object vs một alias. Mức 2: `val` chỉ là trường số. Mức 3: `False, True, True`.

**Bài 2:** Mức 1: hai tuyến đi cùng tổng node. Mức 2: A+B và B+A. Mức 3: gặp ở C0.

**Bài 3:** Mức 1: chỉ cần a,b. Mức 2: `while a is not b`. Mức 3: a hết thì nhận headB, b hết thì nhận headA.

**Bài 4:** Mức 1: giải bài gốc trước. Mức 2: kết quả có thể None. Mức 3: `node.val if node is not None else None`.

**Bài 5:** Mức 1: test identity. Mức 2: dùng chung một đuôi, không copy. Mức 3: kiểm `actual is expected` với ít nhất 5 cấu trúc.

**Bài 6:** Mức 1: độ dài riêng gây lệch. Mức 2: đếm m,n. Mức 3: cho head của list dài đi trước `abs(m-n)` bước.

**Cách tự đánh giá:** Nếu mức 1 đã đủ để bạn làm được, hãy che mức 2–3. Nếu phải dùng mức 3, thử giải lại từ đầu sau khi đóng notebook. Học để có thể **tự suy ra**, không phải chép một đoạn code đã biết.


## 17. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

Phần này dùng **tên hàm `answer_*` khác với `exercise_*`** để notebook vẫn chạy an toàn. Đáp án không cần giống hệt từng ký tự code của bạn, nhưng phải giữ đúng điều kiện và chứng minh cùng độ phức tạp.

Đáp án bài 3 phải trả đúng **chính node** mà hai danh sách chia sẻ. Không dùng `id(...)` để làm khóa lưu trữ trong lời giải; `is` là đủ.


In [40]:
def answer_intersection_160(headA,headB):
    a,b=headA,headB
    while a is not b:
        a=a.next if a is not None else headB
        b=b.next if b is not None else headA
    return a

a,b,c=make_case([1,2],[3],[9,10])
assert answer_intersection_160(a,b) is c
print('Đáp án bài 3: PASS')


Đáp án bài 3: PASS


### Đáp án bài 4 — suy ra từ biến thể

Biến thể trả giá trị là hai bước: nhận node bằng giải gốc, rồi **mới** lấy thuộc tính `val` nếu node tồn tại. Cách này không làm mất các thông tin của danh sách và không làm đổi code nộp LeetCode.

Hãy so sánh với phần bạn tự viết: code nào dễ nhìn ra bất biến hơn? Cả hai đúng vẫn có thể khác nhau về chi phí bộ nhớ hoặc sự rõ ràng.


In [41]:
def answer_first_value_160(a,b):
    node=answer_intersection_160(a,b)
    return node.val if node is not None else None

a,b,c=make_case([],[1],[0])
assert answer_first_value_160(a,b)==0
print('Đáp án bài 4: PASS')


Đáp án bài 4: PASS


### Đáp án bài 6, và cách kiểm tra bài 5

Cách căn chiều dài: đếm m,n, đưa con trỏ trên danh sách dài tiến `abs(m-n)` bước; từ thời điểm đó hai con trỏ cùng cách node giao một số bước nếu có giao. Đáp án cũng đúng khi một list rỗng hoặc hai head chung object.

Mỗi test có một mục đích. Khi tự thiết kế test, ghi chú ca nào kiểm điều kiện dừng, ca nào kiểm cập nhật, ca nào kiểm dữ liệu hợp lệ khác thường.


In [42]:
def answer_align_length_160(headA,headB):
    def length(p):
        n=0
        while p is not None:n+=1;p=p.next
        return n
    m,n=length(headA),length(headB)
    a,b=headA,headB
    for _ in range(max(0,m-n)):a=a.next
    for _ in range(max(0,n-m)):b=b.next
    while a is not b:
        a=a.next;b=b.next
    return a

for la in range(8):
    for lb in range(8):
        for lc in range(5):
            a,b,c=make_case([0]*la,[0]*lb,[0]*lc)
            assert answer_align_length_160(a,b) is c
print('Đáp án bài 6: PASS 320 cấu trúc')


Đáp án bài 6: PASS 320 cấu trúc


### Đáp án / lời giải thích bằng lời cho cả 6 bài

**Bài 1:** `x is y` là False, `x is z` True, và `x.val == y.val` True. `z=x` không tạo node mới.

**Bài 2:** Hai con trỏ đi A+B và B+A, qua phần riêng theo hai thứ tự nhưng cuối cùng gặp ở C0. Nên so identity, không dùng chỉ số trực tiếp.

**Bài 3:** Chọn trạng thái tối thiểu như `solve_*`; đọc lại cell đáp án ngay trên và giải thích từng phép gán.

**Bài 4:** Trả `node.val` chỉ khi node khác None, không thay thế output node trong bài gốc.

**Bài 5:** Đủ ca: cả hai rỗng, giao ngay đầu, giao ở đuôi, không giao cùng số, chênh độ dài hai prefix.

**Bài 6:** Sau khi căn độ dài, hai con trỏ cách cuối bằng nhau; đi song song O(m+n), O(1).

> Nếu chưa thể tự diễn giải được các dòng `return`, hãy quay lại bảng trace và lặp lại bằng một input tự chọn.


## 18. Tổng kết — template suy nghĩ + checklist nộp bài

### Template nhớ nhanh cho #160

1. **Định nghĩa:** Trả object chung đầu tiên của hai singly linked lists, hoặc None; không sửa liên kết.
2. **Trạng thái:** Hai tham chiếu a và b di chuyển trên hai tuyến A+B và B+A.
3. **Quy tắc cập nhật:** Đến None thì đổi sang đầu danh sách còn lại; nếu còn node thì đi .next.
4. **Bất biến:** Hai con trỏ đi cùng số bước; quãng đường thứ hai bù chênh phần riêng ban đầu.
5. **Điều kiện dừng:** Khi a is b: đó là node giao hoặc cả hai None.
6. **Độ phức tạp:** Time O(m+n), extra space O(1).

### Checklist tự đánh giá
- [ ] Tôi phát biểu được đề, output và ràng buộc mà không nhìn code.
- [ ] Tôi hiểu cú pháp mới và tự chạy được ít nhất hai thí nghiệm nhỏ.
- [ ] Tôi trình bày được baseline, điểm yếu và lý do cải tiến.
- [ ] Tôi tự trace được 4 bước, kể cả một nhánh đặc biệt.
- [ ] Tôi phân biệt được chứng minh thuật toán với kiểm thử.
- [ ] Tôi biết khai báo `class Solution` đúng tên hàm, đúng kiểu trả về.
- [ ] Tôi đã làm 6 bài tự luyện **trước khi** xem đáp án.
- [ ] Tôi kiểm tra lại edge cases, time/space, không để print debug khi submit.

**Câu hỏi cuối:** Nếu thay input và biến thể, tôi có tự viết lại thuật toán mà không nhìn notebook được không? Nếu có, bạn đã hiểu kỹ thuật, không chỉ thuộc code.
